In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

d:\projects\Supervised-Learning-Experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("imdb") 
print(np.unique(np.array(dataset["train"]["label"])))

[0 1]


In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

tokenized = dataset.map(tokenize, batched=True)

In [5]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
metric = evaluate.load("accuracy")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8021.32it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [6]:
# ── 4. Define Accuracy Metric ─────────────────────────────────────────────────
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [7]:
# ── 5. Train + Evaluate ───────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"].shuffle().select(range(2000)),
    eval_dataset=tokenized["test"].select(range(500)),
    compute_metrics=compute_metrics,
)

trainer.train()  

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.393280,0.822000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]


TrainOutput(global_step=125, training_loss=0.44855914306640626, metrics={'train_runtime': 30.0323, 'train_samples_per_second': 66.595, 'train_steps_per_second': 4.162, 'total_flos': 263111055360000.0, 'train_loss': 0.44855914306640626, 'epoch': 1.0})

In [8]:
# ── 6. Predict on Custom Text ─────────────────────────────────────────────────
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

my_reviews = [
    "This movie was absolutely fantastic, I loved every second!",
    "What a terrible waste of time. Boring and predictable.",
    "It was okay, nothing special.",
]

for review, result in zip(my_reviews, classifier(my_reviews)):
    print(f"Review: {review}")
    print(f"  → {result['label']} (confidence: {result['score']:.2%})\n")

Review: This movie was absolutely fantastic, I loved every second!
  → LABEL_1 (confidence: 94.99%)

Review: What a terrible waste of time. Boring and predictable.
  → LABEL_0 (confidence: 95.99%)

Review: It was okay, nothing special.
  → LABEL_0 (confidence: 65.30%)



In [9]:
# ── 7. Save the Model ─────────────────────────────────────────────────────────
model.save_pretrained("./my_bert_sentiment")
tokenizer.save_pretrained("./my_bert_sentiment")
print("Model saved to ./my_bert_sentiment")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]

Model saved to ./my_bert_sentiment


In [28]:
question = input("review: ")
rate = classifier(question)
print(f"review: {question} |\t" + "positive" if rate[0]["label"] == "LABEL_1" else "negative")

review: nice |	positive
